<a href="https://colab.research.google.com/github/ultimatepin/card_recognizer/blob/main/notebooks/03_automatic%20crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

colab is a cloud computer

! means a normal terminal command

## v1


In [ ]:
%cd /content

!rm -rf card_recognizer
!git clone https://github.com/ultimatepin/card_recognizer.git

%cd /content/card_recognizer

!pip -q install transformers pillow

/content
Cloning into 'card_recognizer'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 75 (delta 19), reused 34 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 13.91 MiB | 26.32 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/card_recognizer


In [ ]:
import os
import torch

from PIL import Image
from transformers import AutoProcessor, CLIPVisionModelWithProjection

MODEL_NAME = "openai/clip-vit-base-patch32"

# processor converts image to processable format
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# transformer and outputs embedding
model = CLIPVisionModelWithProjection.from_pretrained(MODEL_NAME)

# eval mode not train mode
model.eval()

def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")

    # convert to pytorch tensor(pixel_values) to input model
    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    # not training the model, faster calc
    with torch.inference_mode():
        # pixel_values -> bunch of outputs
        output = model(**inputs)

    # embedding extraction
    embedding = output.image_embeds

    # Normalize the vector
    embedding = embedding / embedding.norm(dim=-1, keepdim=True)

    return embedding

card_folder = "cards"

card_embeddings = {}

for filename in os.listdir(card_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(card_folder, filename)

        card_embeddings[filename] = get_embedding(path)

print(f"Loaded {len(card_embeddings)} cards.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0

Loaded 12 cards.


`get_embedding()` returns a single embedding vector corresponding to `image_path`.

Cosine similarity of two images is calculated using dot product/norm, higher cos value means higher similarity, meaning the angle between vectors determines similarity.

## Recognize

In [ ]:
def recognize(image_path, top_k=3):
    query_embedding = get_embedding(image_path)

    results = []

    for card_name, card_embedding in card_embeddings.items():
        similarity = torch.sum(
            query_embedding * card_embedding
        ).item()

        results.append((card_name, similarity))

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results[:top_k]

In [ ]:
recognize("queries/OGN-249_1.jpg")

[('OGN-265.png', 0.6433490514755249),
 ('OGN-253.png', 0.6378507614135742),
 ('OGN-249.png', 0.6262432336807251)]

In [ ]:
tests = {
    "queries/OGN-249_1.jpg": "OGN-249.png",
    "queries/OGN-249_2.jpg": "OGN-249.png",
    "queries/OGN-253_1.jpg": "OGN-253.png",
    "queries/OGN-255_1.jpg": "OGN-255.png",
    "queries/OGN-265_1.jpg": "OGN-265.png",
}

correct = 0

for image_path, expected in tests.items():
    prediction = recognize(image_path, top_k=1)[0][0]

    print(
        image_path,
        "expected:", expected,
        "predicted:", prediction
    )

    if prediction == expected:
        correct += 1

accuracy = correct / len(tests)

print("Accuracy:", accuracy)

queries/OGN-249_1.jpg expected: OGN-249.png predicted: OGN-265.png
queries/OGN-249_2.jpg expected: OGN-249.png predicted: OGN-265.png
queries/OGN-253_1.jpg expected: OGN-253.png predicted: OGN-265.png
queries/OGN-255_1.jpg expected: OGN-255.png predicted: OGN-255.png
queries/OGN-265_1.jpg expected: OGN-265.png predicted: OGN-265.png
Accuracy: 0.4


In [ ]:
for image_path, expected in tests.items():
    results = recognize(image_path, top_k=3)

    print("\nQuery:", image_path)
    print("Expected:", expected)

    for card_name, score in results:
        print(f"{card_name:15} {score:.4f}")


Query: queries/OGN-249_1.jpg
Expected: OGN-249.png
OGN-265.png     0.6433
OGN-253.png     0.6379
OGN-249.png     0.6262

Query: queries/OGN-249_2.jpg
Expected: OGN-249.png
OGN-265.png     0.6975
OGN-253.png     0.6839
OGN-247.png     0.6775

Query: queries/OGN-253_1.jpg
Expected: OGN-253.png
OGN-265.png     0.7053
OGN-253.png     0.6674
OGN-257.png     0.6530

Query: queries/OGN-255_1.jpg
Expected: OGN-255.png
OGN-255.png     0.7148
OGN-269.png     0.7116
OGN-247.png     0.6831

Query: queries/OGN-265_1.jpg
Expected: OGN-265.png
OGN-265.png     0.7414
OGN-247.png     0.6976
OGN-269.png     0.6861


Results showed that top 1 accuracy is 40%, top 3 accuracy is 80%, and similarity difference is small, so CLIP can't confidently determine the image. Cropping it may give a better accuracy.

In [ ]:
tests = {
    "queries_cropped/OGN-249_1.jpg": "OGN-249.png",
    "queries_cropped/OGN-249_2.jpg": "OGN-249.png",
    "queries_cropped/OGN-253_1.jpg": "OGN-253.png",
    "queries_cropped/OGN-255_1.jpg": "OGN-255.png",
    "queries_cropped/OGN-265_1.jpg": "OGN-265.png",
}

correct = 0

for image_path, expected in tests.items():
    prediction = recognize(image_path, top_k=1)[0][0]

    print(
        image_path,
        "expected:", expected,
        "predicted:", prediction
    )

    if prediction == expected:
        correct += 1

accuracy = correct / len(tests)

print("Accuracy:", accuracy)

queries_cropped/OGN-249_1.jpg expected: OGN-249.png predicted: OGN-249.png
queries_cropped/OGN-249_2.jpg expected: OGN-249.png predicted: OGN-249.png
queries_cropped/OGN-253_1.jpg expected: OGN-253.png predicted: OGN-253.png
queries_cropped/OGN-255_1.jpg expected: OGN-255.png predicted: OGN-255.png
queries_cropped/OGN-265_1.jpg expected: OGN-265.png predicted: OGN-265.png
Accuracy: 1.0


In [ ]:
for image_path, expected in tests.items():
    results = recognize(image_path, top_k=3)

    print("\nQuery:", image_path)
    print("Expected:", expected)

    for card_name, score in results:
        print(f"{card_name:15} {score:.4f}")


Query: queries_cropped/OGN-249_1.jpg
Expected: OGN-249.png
OGN-249.png     0.8915
OGN-253.png     0.7931
OGN-265.png     0.7889

Query: queries_cropped/OGN-249_2.jpg
Expected: OGN-249.png
OGN-249.png     0.8917
OGN-253.png     0.7935
OGN-265.png     0.7905

Query: queries_cropped/OGN-253_1.jpg
Expected: OGN-253.png
OGN-253.png     0.8383
OGN-265.png     0.8184
OGN-247.png     0.7899

Query: queries_cropped/OGN-255_1.jpg
Expected: OGN-255.png
OGN-255.png     0.9037
OGN-251.png     0.8209
OGN-247.png     0.8090

Query: queries_cropped/OGN-265_1.jpg
Expected: OGN-265.png
OGN-265.png     0.8777
OGN-269.png     0.8063
OGN-247.png     0.7771


100% top 1 accuracy with much better similarity scores, proving the main problem was indeed backgronud contamination.

Hence, card preprocessing is an important part of the recognizer.